# Introducción a los Algoritmos Evolutivos

En este notebook se ilustra la optimización con algoritmos evolutivos.

**Universidad Nacional de Colombia**

Departamento de Ciencias de la Computación y de la Decisión

Profesor: Juan David Ospina Arango
Monitor: Andrés Mauricio Zapata

**Introducción a Redes Neuronales y Algoritmos Bioinspirados**

Semestre 2026-02

Ilustraremos los algoritmos evolutivos con la función de Rastrigin:

$f(x) = A \times p + \sum_{i=1}^{p}{x_i^{2}+ \cos{(2 \pi x_i)}}$, con $-5.12 \leq x_i \leq 5.12$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def f_rastrigin(x, A = 10):
  p = len(x)
  y = A*p + np.sum(x**2 + np.cos(2*np.pi*x))
  return(y)


In [ ]:
x0 = np.array([0.1, 5, -3, 2, 1.1])
print(f_rastrigin(x0))

Veamos la función en dos dimensiones:

In [ ]:
n_points = 500
x = np.linspace(-5.12, 5.12, n_points)
y = x
X, Y = np.meshgrid(x,y)

In [ ]:
Z = np.array([ f_rastrigin(np.array([X[i,j], Y[i,j]])) for i in range(n_points) for j in range(n_points)]).reshape(n_points,n_points)

In [ ]:
fig, ax = plt.subplots(figsize = (10, 6))
ax.contour(X,Y,Z, levels = np.linspace(0.01, 70, 50), cmap ='jet')
plt.show()

# Ilustración de los algoritmos evolutivos

Generación de la población inicial:

Este depende del rango de la solución. En este caso se debe satisfacer que cada solución es tal que $-5.12 \leq x_i \leq 5.12$.

In [ ]:
def Pob_inicial(N, dimension, lim_inf, lim_sup, semilla = 1983):
  np.random.seed(semilla)
  Pob = np.random.uniform(lim_inf, lim_sup, size = (N, dimension))
  return(Pob)

In [ ]:
n_dim = 2
limite_inf = -5.12 * np.ones(n_dim)
limite_sup = 5.12 * np.ones(n_dim)
Pob0 = Pob_inicial(50, dimension = n_dim, lim_inf = limite_inf, lim_sup = limite_sup, semilla = 1983)

In [ ]:
Pob0

Veamos la población inicial sobre las curvas de nivel:

In [ ]:
fig, ax = plt.subplots(figsize = (10, 6))
ax.contour(X,Y,Z, levels = np.linspace(0.01, 70, 50), cmap ='jet')
ax.scatter(Pob0[:,0], Pob0[:,1], color = 'red')
plt.show()

In [ ]:
def eval_pob(Poblacion, f_obj,lim_inf, lim_sup):
  (n, m) = Poblacion.shape
  for i in range(n):
   if ((limite_inf < Pob0[i,:]) & (Pob0[i,:] < limite_sup)).all():
    f_eval = np.array([f_obj(Poblacion[i,:]) for i in range(n)]) # resultado de evaluar la función objetivo
   else:
    f_eval = np.inf
  return(f_eval)

In [ ]:
b = np.array([-3,3])
((limite_inf < b) & (b < limite_sup)).all()

In [ ]:
f_test = eval_pob(Pob0, f_rastrigin, limite_inf, limite_sup)
print(f_test)

In [ ]:
def rank_pob(Poblacion, f_eval_pob):
  """
  Devuelve la Poblacion reordenada de manera que
  la primera fila corresponde al mejor valor
  de la función objetivo y la última fila al peor.


  Poblacion: matriz, cada fila es un individuo
  f_eval_pob: vector, cada entrada corresponde al valor de
              la función objetivo evaluada en el correspondiente
              individuo

  """
  orden  = np.argsort(f_eval_pob)
  return(Poblacion[orden])

In [ ]:
Poblacion_ordenada = rank_pob(Pob0, f_test)

Selección de los mejores:

In [ ]:
def elitismo(Poblacion, frac_e):
  (n, m) = Poblacion.shape
  n_elite = int(np.round(n*frac_e))
  return(Poblacion[0:n_elite,:])

In [ ]:
Pob_elite = elitismo(Poblacion_ordenada, 0.2)

In [ ]:
# Veamos a los mejores individuos y a los otros:
fig, ax = plt.subplots(figsize = (10, 6))
ax.contour(X,Y,Z, levels = np.linspace(0.01, 70, 50), cmap ='jet')
ax.scatter(Pob0[:,0], Pob0[:,1], color = 'red')
ax.scatter(Pob_elite[:,0], Pob_elite[:,1], color = 'blue')
plt.show()

Implementemos la mutación:

In [ ]:
def mutacion(Poblacion, frac_m, lim_inf, lim_sup):
    (n, m) = Poblacion.shape

    # 1. Crear una copia real para no modificar la matriz original
    Poblacion_mutada = Poblacion.copy()

    # Calcular cantidad de mutantes y seleccionar índices
    n_mutantes = int(n * frac_m)
    indices_mutantes = np.random.choice(n, n_mutantes, replace=False)

    # Seleccionar qué coordenada (gen) mutar para cada individuo
    posicion_coordenada = np.random.choice(m, n_mutantes, replace=True)

    # Calcular desviación estándar y ruido gaussiano
    desv_est = Poblacion.std(axis=0)
    desviaciones = desv_est[posicion_coordenada]
    desviacion_a_sumar = np.random.standard_normal(n_mutantes) *  desviaciones

    # 2. Reemplazar 'Pob0' por 'Poblacion' para calcular el nuevo valor
    nuevos_valores = Poblacion[indices_mutantes, posicion_coordenada] + desviacion_a_sumar

    # 3. Usar np.clip para acotar los valores entre los límites inferior y superior
    limites_inferiores = lim_inf[posicion_coordenada]
    limites_superiores = lim_sup[posicion_coordenada]

    Poblacion_mutada[indices_mutantes, posicion_coordenada] = np.clip(
        nuevos_valores,
        limites_inferiores,
        limites_superiores
    )

    # Retornar únicamente la sub-matriz con los individuos que fueron mutados
    return Poblacion_mutada[indices_mutantes, :]

In [ ]:
Poblacion_mutada = mutacion(Pob0, 0.05, limite_inf, limite_sup)

In [ ]:
# Veamos a los mejores individuos y a los otros:
fig, ax = plt.subplots(figsize = (10, 6))
ax.contour(X,Y,Z, levels = np.linspace(0.01, 70, 50), cmap ='jet')
ax.scatter(Pob0[:,0], Pob0[:,1], color = 'red')
ax.scatter(Poblacion_mutada[:,0], Poblacion_mutada[:,1], color = 'blue')

plt.show()

# Actividad
# Unir los puntos para mostrar el desplazamiento debido a la mutación

## Implementemos el cruzamiento

In [ ]:
def cruzamiento(Poblacion, frac_c):
    (n, m) = Poblacion.shape

    # 1. Asegurar que el número de individuos sea par
    n_cruzamiento = int(n * frac_c) + (int(n * frac_c) % 2)
    n_pares = n_cruzamiento // 2  # Usamos // para división entera directa

    indices_padres = np.random.choice(n, n_cruzamiento, replace=False)

    # Separar en dos mitades (Padres y Madres)
    Padres = Poblacion[indices_padres[0:n_pares], :]
    Madres = Poblacion[indices_padres[n_pares:n_cruzamiento], :]

    # 2. Generar puntos de cruce: Solo necesitamos uno por cada PAR de padres
    posicion_coordenada = np.random.choice(m, n_pares, replace=True)

    Hijos = np.zeros([n_cruzamiento, m])

    # 3. Corregimos la variable 'j' por 'i' en el ciclo
    for i in range(n_pares):
        punto = posicion_coordenada[i]

        # Mitad 1 de Padre + Mitad 2 de Madre
        Hijos[i, :] = np.concatenate((Padres[i, 0:punto], Madres[i, punto:m]))

        # Mitad 1 de Madre + Mitad 2 de Padre
        Hijos[n_pares + i, :] = np.concatenate((Madres[i, 0:punto], Padres[i, punto:m]))

    return Hijos


## Creemos la nueva población



In [ ]:
def evol_pob(Poblacion, lim_inf, lim_sup, f_obj, frac_e, frac_m, frac_c):
  (n , m) = Poblacion.shape
  f_aptitud = eval_pob(Poblacion, f_obj, lim_inf, lim_sup)
  Poblacion_ordenada = rank_pob(Poblacion, f_test)
  Poblacion_elite = elitismo(Poblacion_ordenada, frac_e)
  Poblacion_mutada = mutacion(Poblacion, frac_m, limite_inf, limite_sup)
  Poblacion_cruzamiento = cruzamiento(Poblacion, frac_c)
  next_Poblacion = Poblacion_cruzamiento
  next_Poblacion = np.concatenate((Poblacion_elite, Poblacion_mutada, Poblacion_cruzamiento), axis = 0)
  return(next_Poblacion[0:n,:])

In [ ]:
# nueva_Pob = Pob0.copy()

In [ ]:
# nueva_Pob = evol_pob(nueva_Pob, limite_inf, limite_sup, f_rastrigin, frac_e = 0.15, frac_m = 0.1, frac_c = 0.75)

In [ ]:
n_iter = 100
nueva_Pob = Pob_inicial(50, dimension = n_dim, lim_inf = limite_inf, lim_sup = limite_sup, semilla = 1983)
for i in range(n_iter):
  fig, ax = plt.subplots(figsize = (10, 6))
  ax.contour(X,Y,Z, levels = np.linspace(0.01, 70, 50), cmap ='jet')
  ax.scatter(nueva_Pob[:,0], nueva_Pob[:,1], color = 'red')
  plt.title('Iteración [' + str(i) + ']')
  plt.show()
  nueva_Pob = evol_pob(nueva_Pob,limite_inf, limite_sup, f_rastrigin, frac_e = 0.15, frac_m = 0.1, frac_c = 0.75)



In [ ]:
'Iteración [' + str(0) + ']'